# Topic 3: S3 Pipeline Patterns

> **Phase 7 → Week 14 → Topic 3**

---

## Interview Cheat Sheet

**Q: What is the small file problem in S3/Spark and how do you solve it?**
> When a Spark job writes many small Parquet files (e.g., 1 file per partition per task, resulting in thousands of 1-10 MB files), subsequent reads become slow because: (1) S3 LIST operations are expensive and slow at scale, (2) each file requires a separate open/close, (3) Parquet footer reads are per-file overhead. Solutions: `coalesce(N)` before write to limit output files, periodic OPTIMIZE on Delta tables, compaction jobs that read+rewrite many small files into large ones.

**Q: What is S3 event-driven architecture and how does it trigger Spark jobs?**
> S3 can emit events (ObjectCreated, ObjectDeleted) to: SNS, SQS, Lambda, EventBridge. Event-driven pattern: new file lands on S3 → S3 notification → SQS queue → Lambda function → triggers EMR step or Glue job. This replaces polling (S3KeySensor) with push-based triggering, reducing latency from minutes to seconds.

**Q: What is S3 Intelligent-Tiering?**
> S3 automatically moves objects between access tiers based on usage patterns — Standard (frequent access) → Infrequent Access → Archive Instant Access → Archive. Zero retrieval fee for Standard/IA tiers; small monitoring fee per object. Ideal for data lake Bronze/Silver where access patterns are unpredictable — you pay for storage, not for guessing what will be accessed.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, shutil

spark = SparkSession.builder \
    .appName("Week14 - S3 Pipeline Patterns") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
BASE = "/tmp/s3_patterns"
if os.path.exists(BASE): shutil.rmtree(BASE)
os.makedirs(BASE)
print("S3 Pipeline Patterns — patterns run locally (S3 paths shown as examples)")

## Part 1: Small File Problem & Compaction

In [ ]:
# Simulate the small file problem
import glob

SMALL_FILES_PATH = f"{BASE}/small_files"
COMPACTED_PATH   = f"{BASE}/compacted"

# Write many tiny batches (simulating streaming micro-batches)
for i in range(20):
    spark.range(100) \
        .withColumn("batch", F.lit(i)) \
        .withColumn("value", F.rand()) \
        .coalesce(1) \
        .write.mode("append") \
        .parquet(f"{SMALL_FILES_PATH}/batch={i}/")

# Count files
small_files = glob.glob(f"{SMALL_FILES_PATH}/**/*.parquet", recursive=True)
print(f"Small files: {len(small_files)} files")

# Read + measure
df_small = spark.read.parquet(SMALL_FILES_PATH)
print(f"Total rows: {df_small.count()}")
print(f"Partitions (tasks): {df_small.rdd.getNumPartitions()}")

# Compaction: read all → coalesce → write
df_small \
    .coalesce(2) \
    .write.mode("overwrite") \
    .parquet(COMPACTED_PATH)

compacted_files = glob.glob(f"{COMPACTED_PATH}/*.parquet")
print(f"\nAfter compaction: {len(compacted_files)} files")

df_compact = spark.read.parquet(COMPACTED_PATH)
print(f"Partitions after compaction: {df_compact.rdd.getNumPartitions()}")
print(f"Total rows (same): {df_compact.count()}")

## Part 2: Partitioning Strategy

In [ ]:
ORDERS_PATH = f"{BASE}/orders_partitioned"

orders = spark.createDataFrame([
    ("ORD001", "2024-01-15", "US",  250.0),
    ("ORD002", "2024-01-15", "EU",  180.0),
    ("ORD003", "2024-01-16", "US",  420.0),
    ("ORD004", "2024-01-16", "APAC", 95.0),
    ("ORD005", "2024-01-17", "US",  310.0),
], ["order_id", "date", "region", "amount"])

# Write partitioned by date (most common partition key)
orders.write \
    .partitionBy("date") \
    .mode("overwrite") \
    .parquet(ORDERS_PATH)

# Partition pruning: only reads date=2024-01-15 partition
jan15 = spark.read.parquet(ORDERS_PATH).filter(F.col("date") == "2024-01-15")
jan15.show()
jan15.explain()  # No "Filter" on date → it's pruned at partition level

print("""
Partitioning strategy rules:
  ✓ Partition by date (most queries filter by date)
  ✓ Partition granularity: year/month for low-volume, date for medium, date+region for large
  ✗ Don't partition by high-cardinality columns (user_id → millions of partitions)
  ✗ Don't partition by more than 2-3 columns (nested partition explosion)
  ✓ Target: 100MB-1GB per partition file (not too many small partitions)
  ✓ For streaming: partition by date+hour to contain file count
""")

## Part 3: S3 Event-Driven Architecture

In [ ]:
print("""
S3 Event-Driven Pipeline Architecture:
══════════════════════════════════════════════════════════════

Pattern 1: S3 → SQS → Lambda → Trigger Glue/EMR

  # 1. Enable S3 event notifications (boto3):
  s3 = boto3.client('s3')
  s3.put_bucket_notification_configuration(
    Bucket='my-data-bucket',
    NotificationConfiguration={
      'QueueConfigurations': [{
        'QueueArn': 'arn:aws:sqs:us-east-1:123:my-queue',
        'Events': ['s3:ObjectCreated:*'],
        'Filter': {'Key': {'FilterRules': [
            {'Name': 'prefix', 'Value': 'raw/orders/'},
            {'Name': 'suffix', 'Value': '.parquet'}
        ]}}
      }]
    }
  )

  # 2. Lambda function triggered by SQS:
  def lambda_handler(event, context):
      for record in event['Records']:
          body = json.loads(record['body'])
          s3_key = body['Records'][0]['s3']['object']['key']
          date = s3_key.split('/')[2]  # raw/orders/2024-01-15/data.parquet

          # Trigger Glue job
          glue = boto3.client('glue')
          glue.start_job_run(
              JobName='orders-etl',
              Arguments={'--date': date}
          )

Pattern 2: S3 → EventBridge → Step Functions

  EventBridge rule:
  {
    "source": ["aws.s3"],
    "detail-type": ["Object Created"],
    "detail": {
      "bucket": {"name": ["my-data-bucket"]},
      "object": {"key": [{"prefix": "raw/orders/"}]}
    }
  }
  Target: Step Functions state machine
  → Automatically starts the ETL workflow on file arrival

Event-driven vs polling (Airflow S3Sensor):
  Polling:      check S3 every N seconds → N-second latency, wasted API calls
  Event-driven: S3 notifies immediately → < 1 second latency, pay-per-event
  Use polling:  when Airflow is your only orchestrator, latency not critical
  Use events:   when latency < 1 min required, or S3 API cost is a concern
══════════════════════════════════════════════════════════════
""")

## Part 4: S3 Storage Optimization

In [ ]:
print("""
S3 Storage Cost Optimization:
══════════════════════════════════════════════════════════════

Storage classes (cheapest to most accessible):
  Standard:             $0.023/GB-month, immediate access
  Standard-IA:          $0.0125/GB-month, $0.01/GB retrieval, < monthly access
  One Zone-IA:          $0.01/GB-month, single AZ (data loss risk)
  Intelligent-Tiering:  auto-moves between Standard/IA/Archive based on access
  Glacier Instant:      $0.004/GB-month, millisecond retrieval, quarterly access
  Glacier Flexible:     $0.0036/GB-month, 1-12 hour retrieval (archival)
  Glacier Deep Archive: $0.00099/GB-month, 12-48 hour retrieval (long-term)

S3 Lifecycle rules (automatic tiering):
  # JSON lifecycle policy:
  {
    "Rules": [
      {
        "ID": "BronzeDataLifecycle",
        "Filter": {"Prefix": "bronze/"},
        "Status": "Enabled",
        "Transitions": [
          {"Days": 30,  "StorageClass": "STANDARD_IA"},    # after 30 days
          {"Days": 90,  "StorageClass": "GLACIER_IR"},      # after 90 days
          {"Days": 365, "StorageClass": "DEEP_ARCHIVE"}    # after 1 year
        ],
        "Expiration": {"Days": 2555}  # delete after 7 years (compliance)
      },
      {
        "ID": "TempDataCleanup",
        "Filter": {"Prefix": "tmp/"},
        "Expiration": {"Days": 3}   # delete temp files after 3 days
      }
    ]
  }

Format optimization for cost:
  ✓ Parquet + Snappy: columnar (read only needed columns), compressed
    → 5-10× smaller than CSV, 3-5× faster to query
  ✓ Target 128-512 MB per file (one file per Spark partition ideal)
  ✗ Avoid CSV/JSON for large datasets (no column pruning)
  ✗ Avoid many small files (S3 LIST calls at $0.005/1000 requests)

Multipart upload (mandatory for files > 100 MB):
  spark.conf.set("spark.hadoop.fs.s3a.multipart.size", "134217728")  # 128 MB
  spark.conf.set("spark.hadoop.fs.s3a.fast.upload", "true")          # stream to S3
  spark.conf.set("spark.hadoop.fs.s3a.threads.max", "64")            # concurrent parts
══════════════════════════════════════════════════════════════
""")

## Part 5: Idempotent S3 Write Patterns

In [ ]:
# Idempotent write patterns — safe to re-run
OUTPUT = f"{BASE}/idempotent_output"

data = spark.createDataFrame([
    ("2024-01-15", "US", 5000.0),
    ("2024-01-15", "EU", 3200.0),
], ["date", "region", "revenue"])

# Pattern 1: mode=overwrite (safe for full daily loads)
data.write \
    .mode("overwrite") \
    .partitionBy("date") \
    .parquet(OUTPUT)

# Pattern 2: overwrite only specific partitions (dynamic partition overwrite)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

new_data = spark.createDataFrame([
    ("2024-01-15", "US", 5500.0),   # updated US revenue
    ("2024-01-16", "US", 4800.0),   # new day
], ["date", "region", "revenue"])

new_data.write \
    .mode("overwrite") \
    .partitionBy("date") \
    .parquet(OUTPUT)
# With dynamic overwrite: only overwrites date=2024-01-15 and inserts 2024-01-16
# date=2024-01-15/EU is gone (entire partition replaced)

print("After dynamic partition overwrite:")
spark.read.parquet(OUTPUT).orderBy("date", "region").show()

print("""
Idempotent write patterns summary:

  mode=overwrite:              replaces all output (safe, but wipes everything)
  partitionOverwriteMode=dynamic: overwrites only affected partitions
  Delta MERGE INTO:            row-level upsert (most precise)
  Prefix with date in path:    s3://bucket/output/date=2024-01-15/ → overwrite that path only

Atomic write with S3 committer:
  Without committer: Spark writes parts files, renames them → S3 has no atomic rename
  With magic committer: files are PUT directly to final location → atomic
  Enable: spark.hadoop.fs.s3a.committer.name=magic
          spark.hadoop.fs.s3a.committer.magic.enabled=true
""")

## Exercises

1. Write a compaction job in PySpark that reads all Parquet files from `s3://bucket/raw/orders/date=*/` and rewrites them as `s3://bucket/compact/orders/date=*/` with at most 2 files per date partition. Make it idempotent.
2. Design an S3 bucket lifecycle policy for a data lake with three zones: Bronze (raw, keep 2 years), Silver (validated, keep 5 years), Gold (aggregated, keep 7 years). Bronze is rarely accessed after 30 days, Silver after 90 days.
3. What is the difference between S3's `s3://`, `s3n://`, and `s3a://` connectors in Spark? Why should you always use `s3a://` and what configurations enable best performance?
4. Implement a Lambda function that triggers a Glue ETL job when a new Parquet file arrives in `s3://bucket/raw/orders/`. Extract the date from the S3 key and pass it as a Glue job argument.
5. You have a Spark job that writes a 10 GB DataFrame to S3 in 5 minutes, producing 2000 small files (5 MB each). The next Spark job that reads this data runs in 25 minutes but should take 3. Diagnose and fix.